# Part 6 — Create Image Parquet

Converts JPG images from `data/images/{ASIN}.jpg` into the HuggingFace parquet format
that notebook `02_cluster_centroid_products.ipynb` expects.

**Uses ALL 1,775 ASINs from both train and val splits** — the image parquet is just
a lookup table for cluster visualization, not related to the 80/20 modeling split.

**Output:** `data/amzn_shoes_monthly_avg_long_ffill_28_7_images/`
```
train-00000-of-00004-part000.parquet
train-00000-of-00004-part001.parquet
train-00000-of-00004-part002.parquet
train-00000-of-00004-part003.parquet
train-00001-of-00004-part000.parquet
...  (16 files total)
train-00003-of-00004-part003.parquet
```

## ② Config

In [39]:
import os
import math
import pandas as pd
from pathlib import Path

def _find_project_root() -> Path:
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / 'data_preparation').exists() and (p / 'code').exists():
            return p
    raise RuntimeError(
        'Cannot find project root.\n'
        'Expected a directory containing both "data_preparation/" and "code/".\n'
        f'Searched from: {Path.cwd()}'
    )

# ── BRAND BUCKET — change this line for each run ───────────────────────
# Valid: women_top1_dream_pairs, women_top2_idifu, women_other,
#        men_top1_bruno_marc,    men_top2_nike,    men_other
BRAND_BUCKET = 'men_asics'

PROJECT_ROOT = _find_project_root()
DATA_ROOT    = PROJECT_ROOT / 'data' / BRAND_BUCKET
DATA_DIR     = DATA_ROOT / 'amzn_shoes_monthly_diffs_ffill_fixed_splits'
IMG_DIR      = PROJECT_ROOT / 'data' / 'images'    # shared (ASIN-unique)
PRED_DIR     = DATA_ROOT / 'predictions'
KEY_DIR      = DATA_ROOT / 'keys'
CLEAN_FILE   = DATA_DIR / 'all_products_clean.parquet'

# BUG FIX: v1 lost OUT_DIR definition (v2 had it). Restored, now bucket-aware.
OUT_DIR = DATA_ROOT / 'amzn_shoes_monthly_avg_long_ffill_28_7_images'

# Paper format: 4 shards x 4 parts = 16 parquet files
N_SHARDS = 4
N_PARTS  = 4

OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f'BRAND_BUCKET : {BRAND_BUCKET}')
print(f'✅ Output folder ready')
print(f'   {OUT_DIR}')

BRAND_BUCKET : men_asics
✅ Output folder ready
   /Users/yvony/PycharmProjects/Amazon_demand_analysis_seperate/data/men_asics/amzn_shoes_monthly_avg_long_ffill_28_7_images


## ③ Get ALL unique ASINs from both train and val

In [40]:
# Read BOTH splits — 80/20 split is for modeling only
# Here we need ALL ASINs so notebook 02 can look up any product photo
df_train = pd.read_parquet(DATA_DIR / 'train-00000-of-00001.parquet')
df_val   = pd.read_parquet(DATA_DIR / 'validation-00000-of-00001.parquet')

all_asins = pd.concat([df_train, df_val])['ASIN'].unique().tolist()

print(f'ASINs from train split:   {df_train["ASIN"].nunique()}')
print(f'ASINs from val split:     {df_val["ASIN"].nunique()}')
print(f'Total unique ASINs (all): {len(all_asins)}')

# Check image coverage
asins_with_images = [a for a in all_asins if (IMG_DIR / f'{a}.jpg').exists()]
asins_missing     = [a for a in all_asins if not (IMG_DIR / f'{a}.jpg').exists()]

print(f'\nASINs with images:    {len(asins_with_images)}')
print(f'ASINs missing images: {len(asins_missing)}')
if asins_missing:
    print(f'Missing examples: {asins_missing[:5]}')

FileNotFoundError: [Errno 2] No such file or directory: '/Users/yvony/PycharmProjects/Amazon_demand_analysis_seperate/data/men_asics/amzn_shoes_monthly_diffs_ffill_fixed_splits/train-00000-of-00001.parquet'

## ④ Convert JPGs to parquet (paper format)

In [ ]:
import numpy as np

total_asins = len(all_asins)
total_files = N_SHARDS * N_PARTS  # 16 files

# Split all ASINs into exactly `total_files` near-even groups.
# np.array_split guarantees 16 groups; for any bucket with >= 16 ASINs (all of
# them have >= 100) none are empty, so all 16 parquet files are always written.
# The old ceil-slice scheme front-loaded the chunks and left the last group(s)
# empty for SMALL buckets (e.g. men_top2_nike=103, women_top2_idifu=178 -> only
# 15 files), which then broke the readback below AND notebook 02 — both load a
# hard-coded list of all 16 filenames.
chunks = [list(map(str, c)) for c in np.array_split(all_asins, total_files)]
chunk_size = max(len(c) for c in chunks)

print(f'Total ASINs:         {total_asins}')
print(f'Total parquet files: {total_files}')
print(f'ASINs per file:      ~{chunk_size}')
print()

file_idx = 0
for shard in range(N_SHARDS):
    for part in range(N_PARTS):
        chunk_asins = chunks[file_idx]
        file_idx += 1

        rows = []
        for asin in chunk_asins:
            img_path = IMG_DIR / f'{asin}.jpg'
            if os.path.exists(img_path):
                with open(img_path, 'rb') as f:
                    img_bytes = f.read()
                image_dict = {'bytes': img_bytes, 'path': None}
            else:
                image_dict = {'bytes': b'', 'path': None}

            rows.append({'ASIN': asin, 'image': image_dict})

        if not rows:
            continue

        df_chunk = pd.DataFrame(rows)
        fname    = f'train-0000{shard}-of-00004-part00{part}.parquet'
        out_path = OUT_DIR / fname
        df_chunk.to_parquet(out_path, index=False)

        size_mb = os.path.getsize(out_path) / 1024 / 1024
        print(f'✅ {fname}  ({len(rows)} ASINs, {size_mb:.1f} MB)')

print(f'\n🎉 Done! All 16 parquet files saved.')

## ⑤ Verify — load back exactly as notebook 02 does

In [ ]:
import io
import datasets
from PIL import Image

# Load exactly as notebook 02 does
data_files = {
    'train': [f'train-0000{i}-of-00004-part00{j}.parquet' for i in range(4) for j in range(4)],
}

ds_img = datasets.load_dataset(
    'parquet',
    data_dir=OUT_DIR,
    data_files=data_files,
)

df_img = ds_img['train'].to_pandas()
df_img = df_img.set_index('ASIN')

print(f'Total ASINs loaded: {len(df_img)}')
print(f'Columns: {list(df_img.columns)}')

# Test reading one image exactly as notebook 02 does
test_asin = df_img.index[0]
img = Image.open(io.BytesIO(df_img.loc[test_asin, :].values[0]['bytes']))
print(f'\n✅ Test image OK — ASIN: {test_asin}, Size: {img.size}, Mode: {img.mode}')
print('\n✅ Ready for notebook 02!')